In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime
103259,DEBIT,4,4,-18.08,159.99,Shipping on time,0,48,Water Sports,Chicago,EE. UU.,Andrew,7856,Smith,Consumer,IL,1811 Hazy Oak Subdivision,60621,7,Fan Shop,41.772770,-87.635033,Saint-Pierre-des-Corps,Francia,7856,2015-09-23 06:27:00,18173,1073,40.00,0.20,45423,199.99,-0.11,1,199.99,159.99,-18.08,Western Europe,COMPLETE,1073,48,Pelican Sunstream 100 Kayak,199.99,Standard Class,09-27-2015,06:27
22293,DEBIT,3,4,28.00,100.00,Advance shipping,0,24,Women's Apparel,Caguas,Puerto Rico,Andrew,8415,Garza,Consumer,PR,1233 Honey Village,725,5,Golf,18.225439,-66.370567,MÃ©rida,MÃ©xico,8415,2015-03-31 10:12:00,6127,502,0.00,0.00,15314,50.00,0.28,2,100.00,100.00,28.00,Central America,COMPLETE,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,04-03-2015,10:12
49141,DEBIT,6,2,-58.80,293.98,Late delivery,1,43,Camping & Hiking,Highland,EE. UU.,Samantha,12101,Smith,Corporate,CA,1773 Amber Walk,92346,7,Fan Shop,34.129341,-117.180023,Philadelphia,Estados Unidos,12101,2016-05-22 07:52:00,34755,957,6.00,0.02,86820,299.98,-0.20,1,299.98,293.98,-58.80,East of USA,COMPLETE,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Second Class,05-28-2016,07:52
84139,DEBIT,4,4,2.32,23.24,Shipping on time,0,40,Accessories,Jacksonville,EE. UU.,Eugene,8739,Perry,Consumer,FL,5559 Colonial Prairie Manor,32225,6,Outdoors,30.353422,-81.513092,GÃ³mez Palacio,MÃ©xico,8739,2017-01-31 05:25:00,52148,897,1.75,0.07,130294,24.99,0.10,1,24.99,23.24,2.32,Central America,COMPLETE,897,40,Team Golf New England Patriots Putter Grip,24.99,Standard Class,02-04-2017,05:25
142121,DEBIT,5,2,-155.99,103.99,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Mary,4374,Smith,Home Office,PR,406 Shady Cider Corner,725,4,Apparel,18.253180,-66.370628,Xichang,China,4374,2015-12-11 02:40:00,23574,403,26.00,0.20,58975,129.99,-1.50,1,129.99,103.99,-155.99,Eastern Asia,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Second Class,12-16-2015,02:40


In [5]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [6]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [7]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [8]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [9]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [10]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
22963,22964,18.212028,-66.037056,First Class
22037,22038,18.283058,-66.370598,First Class
7186,7187,26.202797,-98.300934,Second Class
2777,2778,33.905682,-118.160255,Standard Class
17816,17817,32.261726,-106.771316,Standard Class


In [11]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [12]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime,store_id
160972,TRANSFER,6,4,-227.39,272.98,Shipping canceled,0,43,Camping & Hiking,Caguas,Puerto Rico,Andrew,5834,Fernandez,Consumer,PR,6718 High Panda Dell,725,7,Fan Shop,18.272070,-66.370560,Houston,Estados Unidos,5834,2016-07-10 05:31:00,38105,957,27.0,0.09,95114,299.98,-0.83,1,299.98,272.98,-227.39,US Center,SUSPECTED_FRAUD,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Standard Class,07-16-2016,05:31,22700
12481,DEBIT,5,4,67.86,260.98,Late delivery,1,43,Camping & Hiking,Caguas,Puerto Rico,Heather,6928,Smith,Consumer,PR,9394 Merry Towers,725,7,Fan Shop,18.216484,-66.370575,Kinshasa,RepÃºblica DemocrÃ¡tica del Congo,6928,2016-09-05 14:01:00,42034,957,39.0,0.13,104930,299.98,0.26,1,299.98,260.98,67.86,Central Africa,COMPLETE,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Standard Class,09-10-2016,14:01,5616
5832,CASH,2,1,-68.24,97.49,Late delivery,1,18,Men's Footwear,Las Vegas,EE. UU.,Arthur,1768,Smith,Consumer,NV,4661 Iron Landing,89119,4,Apparel,36.085934,-115.172638,Paris,Francia,1768,2015-09-07 07:32:00,17080,403,32.5,0.25,42715,129.99,-0.70,1,129.99,97.49,-68.24,Western Europe,CLOSED,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,First Class,09-09-2015,07:32,3459
12604,DEBIT,5,4,-71.99,79.99,Late delivery,1,9,Cardio Equipment,Caguas,Puerto Rico,Mary,11664,Smith,Consumer,PR,3180 Golden Cider Circle,725,3,Footwear,18.283337,-66.370537,Torquay,Reino Unido,11664,2017-07-26 13:55:00,64229,191,20.0,0.20,160587,99.99,-0.90,1,99.99,79.99,-71.99,Northern Europe,COMPLETE,191,9,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,07-31-2017,13:55,602
109067,PAYMENT,5,4,103.93,296.95,Late delivery,1,17,Cleats,San Antonio,EE. UU.,Diana,254,Hale,Consumer,TX,5950 Silver Elk Ledge,78223,4,Apparel,31.097872,-99.808777,Bristol,Reino Unido,254,2015-09-04 07:21:00,16874,365,3.0,0.01,42172,59.99,0.35,5,299.95,296.95,103.93,Northern Europe,PENDING_PAYMENT,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,09-09-2015,07:21,2969


In [13]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

¡DataFrames del Modelo en Estrella listos en memoria!


In [14]:
host=os.getenv("DB_HOST"),
port=os.getenv("DB_PORT"),
user=os.getenv("DB_USER"),
password=os.getenv("DB_PASS"),
database=os.getenv("DB_NAME")

In [15]:
# 1. Primero conectamos al servidor MySQL sin indicar base de datos (AÑADE LA / AL FINAL)
engine_server = create_engine(f"mysql+pymysql://{user}:{password}@{host}/")

# 2. Creamos la base de datos si no existe (Esto lo tienes perfecto)
with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {database}"))

# 3. Ahora sí nos conectamos a la base de datos específica (Esto también está perfecto)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

OperationalError: (pymysql.err.OperationalError) (2003, 'Can\'t connect to MySQL server on "(\'localhost\',)" ([Errno 11001] getaddrinfo failed)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# Primero conectamos al servidor MySQL sin indicar base de datos
engine_server = create_engine(f"mysql+pymysql://{user}:{password}@{host}/")
 
# Creamos la base de datos si no existe
with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {database}"))
 
# Ahora sí nos conectamos a la base de datos
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

OperationalError: (pymysql.err.OperationalError) (2003, 'Can\'t connect to MySQL server on "(\'127.0.0.1\',)" ([Errno 11001] getaddrinfo failed)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
#creamos el engine a ver que tal
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")
try:
    dim_products.to_sql('products', con=engine, if_exists="replace", index=False)
    print("✅ ¡Conexión exitosa! Los artistas se han guardado en la tabla 'tabla_general'.")
except Exception as e:
    print(f"❌ Ups, algo falló: {e}")
 

ValueError: invalid literal for int() with base 10: 'None'

In [16]:
import os

from dotenv import load_dotenv

from sqlalchemy import create_engine, text
 
load_dotenv()
 
USER    = os.getenv("DB_USER")

PASS    = os.getenv("DB_PASSWORD")

HOST    = os.getenv("DB_HOST")

PORT    = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto

DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}")
 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:

    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))

    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")

print(f"✅ Conectado a '{DB_NAME}'")
 

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)